# DermaCareAI: train on a free GPU (Google Colab)

This notebook trains one DermaCareAI checker on a GPU, which is roughly 10-20x faster than a laptop CPU, using
**larger photos (300 px), EfficientNetB3, test-time augmentation and extra weight for skin cancers**.

**Before you start**
1. On your computer, run `.\scripts\make_colab_bundle.ps1`. It creates `run\dermacare_colab_bundle.zip`, which holds code only (no data or secrets).
2. In Colab, choose **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from top to bottom (Shift+Enter).

**Data licences:** HAM10000 is CC BY-NC 4.0 (non-commercial). SCIN uses the SCIN Data Use License, which prohibits
re-identifying contributors. The data is downloaded inside Colab from the official sources.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload and unpack the project bundle

In [ ]:
from google.colab import files
import zipfile, os
uploaded = files.upload()          # choose run/dermacare_colab_bundle.zip
name = next(iter(uploaded))
os.makedirs('/content/dermacare', exist_ok=True)
zipfile.ZipFile(name).extractall('/content/dermacare')
%cd /content/dermacare
!ls

## 3. Install pinned versions (they match your laptop, so the model loads there)

In [ ]:
!pip install -q "tensorflow==2.21.0" "keras==3.15.1" pyyaml opencv-python-headless scikit-learn pandas matplotlib
import tensorflow as tf, keras
print('TF', tf.__version__, '| Keras', keras.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 4. Choose which checker to train
* `"lesion"`: **mole or spot** (HAM10000, ~2.8 GB download, about 1-2 h on a T4)
* `"general"`: **rash or other skin problem** (SCIN, ~3.9 GB download, about 1 h on a T4)

In [ ]:
CHECKER = "lesion"   # or "general"
CONFIG = "ml/configs/train_gpu.yaml" if CHECKER == "lesion" else "ml/configs/train_scin_gpu.yaml"
OUT = "ml/artifacts" if CHECKER == "lesion" else "ml/artifacts/scin"
print(CHECKER, CONFIG)

## 5. Download and verify the data (official sources, MD5-checked)

In [ ]:
if CHECKER == "lesion":
    !python -m ml.src.download_ham10000
else:
    !python -m ml.src.scin_prepare

## 6. Train, evaluate and export (selection uses the validation set only; the test set is reported once)

In [ ]:
!python -m ml.src.run_pipeline --config $CONFIG

## 7. Show the honest test results

In [ ]:
import json, pathlib
version = pathlib.Path(f"{OUT}/models/LATEST").read_text().strip()
m = json.load(open(f"{OUT}/models/{version}/metrics.json"))["test"]
print("model:", version)
print(f"test photos {m['n_images']} | balanced accuracy {m['balanced_accuracy']:.3f} | macro-F1 {m['macro_f1']:.3f}")
for c, v in m["per_class"].items():
    print(f"  {c:20s} sensitivity {v['recall_sensitivity'] or 0:.2f}  specificity {v['specificity'] or 0:.2f}  F1 {v['f1']:.2f}")

## 8. Download the trained model

In [ ]:
import shutil
archive = shutil.make_archive(f"/content/{version}", "zip", f"{OUT}/models", version)
files.download(archive)

## 9. Install it on your laptop
1. Unzip the downloaded file into `ml\artifacts\models\` (mole) or `ml\artifacts\scin\models\` (rash), so you get a
   folder named after the version.
2. In the project folder run (replace `<version>`):
   ```
   .\.venv\Scripts\flask.exe --app wsgi register-model ml\artifacts\models\<version> --activate
   ```
   For the rash checker use `ml\artifacts\scin\models\<version>`.
3. Restart the website: `.\scripts\restart.ps1`. The About page will show the new test results.